# Women's E-Commerce Clothing Reviews Analysis

**Team:** Tirth Trivedi (IU2441230646) and Dhairya Shah (IU2441230652)

This notebook cleans and analyzes the dataset, creates eight visualizations, and saves a cleaned CSV. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATASET_NAME = "Womens Clothing E-Commerce Reviews.csv"
DATASET_URL = "https://raw.githubusercontent.com/Tirth007Triv/women-ecommerce-clothing-reviews-analysis/main/dataset/Womens%20Clothing%20E-Commerce%20Reviews.csv"
cwd = Path.cwd()
candidates = [cwd/"dataset"/DATASET_NAME, cwd.parent/"dataset"/DATASET_NAME,
              cwd/DATASET_NAME, cwd.parent/DATASET_NAME,
              Path("/content/dataset")/DATASET_NAME, Path("/content")/DATASET_NAME]
file_path = next((p.resolve() for p in candidates if p.is_file()), None)
if file_path is None:
    for base in [cwd, *list(cwd.parents)[:3]]:
        candidate = base/"dataset"/DATASET_NAME
        if candidate.is_file():
            file_path = candidate.resolve()
            break
if file_path is not None:
    df = pd.read_csv(file_path)
    output_dir = file_path.parent
    print(f"Loaded dataset from: {file_path}")
else:
    try:
        df = pd.read_csv(DATASET_URL)
        output_dir = cwd/"dataset"
        output_dir.mkdir(parents=True, exist_ok=True)
        print("Loaded dataset from the project's GitHub repository.")
    except Exception as exc:
        raise FileNotFoundError("Dataset not found locally and download failed. Place the CSV in a folder named 'dataset', or run with an internet connection.") from exc
print("Original shape:", df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))
print("Duplicate rows before removing exported index:", int(df.duplicated().sum()))
display(df.dtypes.to_frame("Data type"))

## Data cleaning

The exported index column is removed if present. Missing review text is replaced with empty strings, missing categories are marked Unknown, numeric columns are converted and validated, and exact duplicates are removed. Plausible IQR outliers are flagged rather than automatically deleted.

In [ ]:
data = df.drop(columns=["Unnamed: 0"], errors="ignore").copy()
for col in ["Title", "Review Text"]:
    if col in data.columns:
        data[col] = data[col].fillna("").astype(str)
for col in ["Division Name", "Department Name", "Class Name"]:
    if col in data.columns:
        data[col] = data[col].fillna("Unknown").astype(str).str.strip()
numeric_cols = ["Clothing ID", "Age", "Rating", "Recommended IND", "Positive Feedback Count"]
missing_cols = [col for col in numeric_cols if col not in data.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data.loc[~data["Age"].between(13, 100), "Age"] = np.nan
data.loc[~data["Rating"].between(1, 5), "Rating"] = np.nan
data.loc[~data["Recommended IND"].isin([0, 1]), "Recommended IND"] = np.nan
data.loc[data["Positive Feedback Count"] < 0, "Positive Feedback Count"] = np.nan
rows_before = len(data)
data = data.dropna(subset=numeric_cols).drop_duplicates().copy()
for col in numeric_cols:
    data[col] = data[col].astype(int)
print("Rows removed during numeric validation / duplicate removal:", rows_before-len(data))
print("Cleaned shape:", data.shape)
print("Missing values remaining:", int(data.isna().sum().sum()))
print("Duplicate rows remaining:", int(data.duplicated().sum()))
display(data.head())

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3-q1
    return q1-1.5*iqr, q3+1.5*iqr
outlier_summary = []
for col in ["Age", "Positive Feedback Count"]:
    low, high = iqr_bounds(data[col])
    flagged = int(((data[col] < low) | (data[col] > high)).sum())
    outlier_summary.append({"Column":col,"Lower bound":low,"Upper bound":high,"Flagged rows":flagged})
display(pd.DataFrame(outlier_summary))
print("IQR flags are reviewed, not automatically removed, because values can be plausible.")

## Eight visualizations

In [ ]:
project_dir = output_dir.parent if output_dir.name.lower() == "dataset" else cwd
graphs_dir = project_dir/"graphs"
graphs_dir.mkdir(parents=True, exist_ok=True)
def save_show(filename):
    plt.tight_layout()
    plt.savefig(graphs_dir/filename, dpi=150, bbox_inches="tight")
    plt.show()

data["Rating"].value_counts().sort_index().plot(kind="bar", figsize=(8,5))
plt.title("Customer Rating Distribution"); plt.xlabel("Rating (1–5)"); plt.ylabel("Reviews"); plt.xticks(rotation=0)
save_show("01_rating_distribution.png")

data["Recommended IND"].map({1:"Recommended",0:"Not recommended"}).value_counts().plot(kind="bar", figsize=(7,5))
plt.title("Recommended vs Not Recommended"); plt.ylabel("Reviews"); plt.xticks(rotation=0)
save_show("02_recommendations.png")

data.groupby("Department Name")["Rating"].mean().sort_values().plot(kind="barh", figsize=(9,5))
plt.title("Average Rating by Department"); plt.xlabel("Average rating"); plt.xlim(0,5)
save_show("03_department_ratings.png")

class_stats = data.groupby("Class Name")["Rating"].agg(["mean","count"])
class_stats = class_stats[class_stats["count"]>=30].sort_values("mean",ascending=False).head(10).sort_values("mean")
class_stats["mean"].plot(kind="barh", figsize=(10,6))
plt.title("Top-Rated Clothing Classes (30+ reviews)"); plt.xlabel("Average rating"); plt.xlim(0,5)
save_show("04_class_ratings.png")

data["Age"].plot(kind="hist", bins=20, figsize=(8,5), edgecolor="black")
plt.title("Customer Age Distribution"); plt.xlabel("Age"); plt.ylabel("Reviews")
save_show("05_age_distribution.png")

data["Positive Feedback Count"].clip(upper=30).plot(kind="hist", bins=16, figsize=(8,5), edgecolor="black")
plt.title("Positive Feedback Counts (30 means 30+)"); plt.xlabel("Feedback count"); plt.ylabel("Reviews")
save_show("06_positive_feedback.png")

data["Review_Length"] = data["Review Text"].str.len()
data.boxplot(column="Review_Length", by="Rating", figsize=(8,5), grid=False)
plt.suptitle(""); plt.title("Review Length by Rating"); plt.xlabel("Rating"); plt.ylabel("Characters")
save_show("07_review_length_by_rating.png")

tokens = re.findall(r"\b[a-z]{3,}\b", " ".join(data["Review Text"].astype(str)).lower())
stop_words = set("the and for that this with was but are you have had not she her his they them their from were will would can all about very just really too its one out get got fit size dress love great good nice wear wore look like more when what which who how did because been into also than then there here still only some much little".split())
word_counts = pd.Series([word for word in tokens if word not in stop_words]).value_counts().head(15).sort_values()
word_counts.plot(kind="barh", figsize=(9,6))
plt.title("Most Frequent Review Words"); plt.xlabel("Count")
save_show("08_common_words.png")
print(f"Eight charts saved in: {graphs_dir.resolve()}")

## Summary findings

In [ ]:
print("Cleaned records:", len(data))
print(f"Mean rating: {data['Rating'].mean():.2f}/5")
print(f"Ratings 4 or 5: {(data['Rating']>=4).mean()*100:.1f}%")
print(f"Recommendation rate: {data['Recommended IND'].eq(1).mean()*100:.1f}%")
print(f"Mean age: {data['Age'].mean():.1f}; median age: {data['Age'].median():.0f}")
print(f"Median positive feedback count: {data['Positive Feedback Count'].median():.0f}")
print(f"Mean review length: {data['Review_Length'].mean():.1f} characters")
print(f"Age-rating correlation: {data['Age'].corr(data['Rating']):.3f}")
display(data.groupby("Department Name")["Rating"].agg(["mean","count"]).sort_values("mean",ascending=False))
cleaned_path = output_dir/"cleaned_womens_clothing_reviews.csv"
data.drop(columns=["Review_Length"], errors="ignore").to_csv(cleaned_path,index=False)
print(f"Cleaned CSV saved to: {cleaned_path.resolve()}")